In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model
from prompts import profile, email_respond, triage_system_prompt, triage_user_prompt, prompt_instructions, agent_system_prompt
from langchain_core.tools import tool

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [3]:
class Router(BaseModel):
    """Analiza el e-mail no leído y enrrútalo de acuerdo con su contenido."""

    reasoning: str = Field(
        description="Raciocinio paso a paso tras la clasificación."
    )
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="La clasificación de un e-mail: `ignore` para e-mails irrelevantes, "
        "`notify` para informaciones importantes que no necesitan de una respuesta, "
        "`respond` para e-mails que requieren una respuesta",
    )

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Escribe y envia un e-mail."""
    # Respuesta de placeholder - en un aplicativo real, enviaria el e-mail
    return f"E-mail enviado para {to} con el asunto {subject}"

@tool
def schedule_meeting(
    attendees: list[str],
    subject: str,
    duration_minutes: int,
    preferred_day: str
) -> str:
    """Programa una reunión en el calendario."""
    return f"Reunión '{subject}' programada para {preferred_day} con {len(attendees)} participantes"

@tool
def check_calendar_availability(day: str) -> str:
    """Verifica la disponibilidad de agenda para una fecha determinada."""
    return f"Horarios disponibles en {day}: 9:00 AM, 2:00 PM, 4:00 PM"

In [4]:
system_prompt = triage_system_prompt.format(
    full_name=profile["full_name"],
    name=profile["name"],
    examples=None,
    user_profile_background=profile["user_profile_background"],
    triage_no=prompt_instructions["triage_rules"]["ignore"],
    triage_notify=prompt_instructions["triage_rules"]["notify"],
    triage_email=prompt_instructions["triage_rules"]["respond"],
)

user_prompt = triage_user_prompt.format(
    author=email_respond["from"],
    to=email_respond["to"],
    subject=email_respond["subject"],
    email_thread=email_respond["body"],
)


In [6]:
llm_router = llm.with_structured_output(Router)
result = llm_router.invoke(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
)

print(result)

reasoning="El correo es de Alice Smith, un miembro del equipo, y contiene una pregunta directa a Sarah Chen sobre la documentación de la API. La regla 'RESPOND' aplica a 'Preguntas directas de miembros del equipo'." classification='respond'


In [7]:
#from langgraph.prebuilt import create_react_agent Obsolote
from langchain.agents import create_agent

def create_prompt(state):
    return [
        {
            "role": "system",
            "content": agent_system_prompt.format(
                instructions=prompt_instructions["agent_instructions"],
                **profile
            ),
        }
    ] + state['messages']

tools=[write_email, schedule_meeting, check_calendar_availability]
agent = create_agent(
    model=llm,
    tools=tools,
    prompt=create_prompt,
)

C:\Users\tcpad\AppData\Local\Temp\ipykernel_25216\164182459.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [21]:
response = agent.invoke(
    {"messages": [
        {
            "role": "user",
            "content": "Cuál es mi disponibilidad para el martes? En la respuesta final debes indicar también el nombre exacto de la herramienta utilizada."
        }
    ]}
)

response["messages"][-1].pretty_print()
response["messages"][1].tool_calls[0]["name"]

In [ ]:
from langgraph.graph import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

def triage_router(state: State) -> Command[
    Literal["response_agent", "__end__"]
]:
    author = state["email_input"]["author"]
    to = state["email_input"]["to"]
    subject = state["email_input"]["subject"]
    email_thread = state["email_input"]["email_thread"]

    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"],
        name=profile["name"],
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
        examples=None,
    )

    user_prompt = triage_user_prompt.format(
        author=author,
        to=to,
        subject=subject,
        email_thread=email_thread,
    )

    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    if result.classification == "respond":
        print("✅ Clasificación: RESPONDER - Este e-mail requiere una respuesta")
        goto = "response_agent"
        update = {
            "messages": [
                {
                    "role": "user",
                    "content": f"Responde al email {state['email_input']}",
                }
            ]
        }
    elif result.classification == "ignore":
        print("✅ Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas")
        update = None
        goto = END
    elif result.classification == "notify":
        print("✅ Clasificación: NOTIFICAR - Este e-mail contiene informaciones importantes")
        update = None
        goto = END
    else:
        raise ValueError(f"Clasificación inválida: {result.classification}")

    return Command(goto=goto, update=update)

In [16]:
import os
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from IPython.display import Image, display

email_agent = StateGraph(State)
email_agent = email_agent.add_node("triage_router", triage_router)
email_agent = email_agent.add_node("response_agent", agent)
email_agent = email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile()
#display(Image(email_agent.get_graph(xray=True).draw_mermaid_png()))

In [17]:
email_input = {
    "author":"Equipo de Marketing <marketing@amazingdeals.com>",
    "to": "Sarah Chen <sarah.chen@company.com>",
    "subject": "�� OFERTA EXCLUSIVA: ¡Descuento por Tiempo Limitado en Herramientas para Desarrolladores!",
    "email_thread": "Estimado(a) Desarrollador(a),\n\n¡No pierda esta oportunidad INCREÍBLE!\n\n�� POR TIEMPO LIMITADO, obtenga un 80% DE DESCUENTO en nuestro Paquete Premium para Desarrolladores\n\n�� RECURSOS:\n- Autocompletado de código revolucionario con IA\n- Entorno de desarrollo basado en la nube\n- Soporte al cliente 24/7\n- ¡Y mucho más!\n\n�� Precio normal: R$ 999/mes\n�� SU PRECIO ESPECIAL: ¡Solo R$ 199/mes!\n\n⏳ ¡Apúrese! Esta oferta expira en:\n¡SOLO 24 HORAS!\n\nHaga clic aquí para canjear su descuento: https://amazingdeals.com/special-offer\n\nAtentamente,\nEquipo de Marketing\n\nPara cancelar la suscripción, haga clic aquí",
}

response = email_agent.invoke({"email_input": email_input})

✅ Clasificación: IGNORAR - Este e-mail puede ser ignorado sin problemas
